# Notebook 12 · ES Arena Prospective Evaluation (v0.3.2 — forecasting vs. provenance separated)

**Market Mood Forecasting · original frozen M1 unchanged · READ-ONLY retrospective evaluation**

> **Purpose:** assess the original first-30 accepted ES cohort using two independent dimensions: (A) *descriptive forecasting performance* on accepted and officially settled outcomes with authentic, valid forecasts, and (B) *strength of provenance certification* based on local model/snapshot/receipt and time evidence. The first is not silently reduced to a one-record subset just because independent server-time verification is unavailable.

**No retraining, no prediction generation, no data purchases, no network access, no existing evidence changes, no Arena submission.** Notebook 08–11, frozen M1, existing forecasts and original receipt files remain unchanged. This notebook does not establish the later 60+60 protocol.

### Explicit populations

1. **Main descriptive, accepted-and-settled cohort:** within the first 30 accepted opportunities, an independently documented official Arena outcome, and either (a) a native forecast with valid class probabilities, consistent original model and source artifacts, hash-linked scored HTTP-201 receipt and no contradictory audit finding, or (b) the original October 20 recovered CLI + official-page acceptance explicitly marked `LEGACY_LIMITATION`. A missing separately verifiable UTC *server* timestamp is disclosed but **does not by itself erase an accepted forecast from this descriptive analysis**.
2. **Strictly time-certified native cohort:** additionally requires the verifier's `VERIFIED_LOCAL_CHAIN` status, including independently supported server acceptance time. This is an optional sensitivity subset; if empty, **no performance statistics are reported for it**.
3. **Excluded/quarantined evidence:** invalid or contradictory original probabilities, corrupt hashes, missing scored acceptance, unverified official outcome, or source-date problems remain in the ledger and coverage funnel, not in main scoring. Never backfill or replace an original eligible opportunity with a favorable later record.

**Timestamp caution:** Arena's raw response `created_at` may be timezone-naive (e.g. `2026-11-11T10:01:33.025476`), while the locally recorded receipt `recorded_at_utc` can be offset-aware (`2026-11-11T10:01:33.303003+00:00`). The latter is *client-observed receipt timing*, **not independently signed server acceptance UTC**. Do not append `Z`, infer server timezone, or upgrade the strict verifier from this resemblance alone.

**No skill/profit claims:** Hash matches prove local byte consistency, not that the model executed or the provider had published each input as-of a particular moment. A small descriptive sample cannot demonstrate a trading edge. Follow actual original Arena outcomes for Arena scoring.

Execution outputs from the prior v0.3 notebook have been cleared; do not treat this file's design notes as fresh evidence of how many challenges are settled.

## 00 · Installation and safeguards

1. Retain v0.1 and v0.3 notebooks unchanged as dated historical versions. Save this as `notebooks/12_es_arena_prospective_evaluation_v0_3_2_separated_evaluation_and_provenance.ipynb`.
2. Continue using the **existing** `verify_es_forecast_provenance.py` (and its 11 passing Windows tests) in the project root; this notebook changes **evaluation selection and disclosure only**, not the verifier's strict standards.
3. Preserve original private model, DBNs, reviewed audits, snapshots, forecasts, attempts, and receipt JSON bytes. Never edit an old receipt's timezone or regenerate a past forecast to make a record pass.
4. Maintain independently checked official outcomes under `data/processed/es_arena_research_v0_1/arena_official_settlements.csv`. The notebook has **no network access** and will not automatically fetch fresh Arena settlements. Peer vectors and dates must also be separately vetted.
5. `WRITE_MISSING_TEMPLATES = False` is the safe default. Use the existing original project virtual environment. Executed notebook output may contain sensitive private filenames and challenge metadata; review before committing or sharing.
6. First run the original read-only provenance CLI (`python verify_es_forecast_provenance.py --root .`), then run this notebook. `INCOMPLETE` due solely to unresolved server UTC time is **not** a corrupt forecast. `FAIL` remains quarantined.

### Independent report labels

- **Receipt-accepted:** server HTTP 201 + `counts_for_score=True` + matching challenge ID/prediction ID + full original forecast SHA-256.
- **Artifact-consistent:** original model hash, canonical dates, snapshot and review evidence, immutable probabilities, and matching manifests pass the existing verifier.
- **Client recorded time:** local `recorded_at_utc`, when present; cannot alone certify server UTC acceptance time.
- **Strict UTC-certified:** verifier marks `VERIFIED_LOCAL_CHAIN`; an unzoned server `created_at` stays unresolved.
- **Legacy October 20:** recovered original console/API and official-page evidence; does not receive invented native artifacts or full provenance status.

The primary descriptive analysis is conditional on the *available verified official settlement CSV*, which may lag behind the Arena site. Its number of rows is not necessarily the currently settled number on the platform.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, hashlib, math, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

CLASSES = ("bearish", "neutral", "bullish")
CLASS_TO_I = {v:i for i,v in enumerate(CLASSES)}
COHORT_START = "2026-10-20"
COHORT_TARGET_N = 30
ORIGINAL_FIRST_CHALLENGE_ID = "118cefc4-96a8-4e47-b617-73af53c67d03"
EXPECTED_CONTRACT = "ESZ6"  # a roll requires documenting/updating the protocol, never silently reinterpreting it
DEAD_ZONE_PCT = 0.3
FROZEN_DEPLOYMENT = "deployment_ab4d5c14423352c7"
FROZEN_MODEL_SHA256 = "5e8a2a9f3ba9a310b340aa09c3ddc823aab2fe9aed728ac87965c2e61a5ae68d"
BOOTSTRAP_SEED = 20261026
WRITE_MISSING_TEMPLATES = False  # ONE-TIME OPT-IN only; never overwrite an existing CSV

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / "es_arena_daily.py").is_file() and (p / "notebooks").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside your Market-Mood-Forecasting checkout; repo root not found")

EVIDENCE = PROJECT_ROOT / "data/processed/es_arena_live"
FORECAST_DIR = EVIDENCE / "forecasts"
RECEIPT_DIR = EVIDENCE / "receipts"
SNAPSHOT_DIR = PROJECT_ROOT / "data/processed/es_arena_nb11_v0_1/snapshots"
MODEL_DIR = PROJECT_ROOT / "data/processed/es_arena_nb10_v0_1" / FROZEN_DEPLOYMENT
RESEARCH_DIR = PROJECT_ROOT / "data/processed/es_arena_research_v0_1"
LEGACY_DIR = RESEARCH_DIR / "legacy_submissions"
OFFICIAL_CSV = RESEARCH_DIR / "arena_official_settlements.csv"
PEERS_CSV = RESEARCH_DIR / "arena_peer_forecasts.csv"
TRAIN_COUNTS_CSV = RESEARCH_DIR / "training_class_counts.csv"
SCHEDULE_CSV = RESEARCH_DIR / "evaluation_calendar.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    from verify_es_forecast_provenance import audit_repository, audit_model, sha256_file as audit_sha256_file
except ImportError as exc:
    raise ImportError("Copy verify_es_forecast_provenance.py into the project root before running this notebook") from exc
print("PROJECT:", PROJECT_ROOT)
print("MODE:", "OPTIONAL EMPTY TEMPLATE CREATION" if WRITE_MISSING_TEMPLATES else "READ-ONLY")
print("Protocol: 30 accepted ES forecasts; model unchanged; official Arena settlement is ground truth")

## 01 · Protocol fingerprint, environment and provenance

The model identity, training cutoff and source deployment are **historical constants**, not values to optimize after observing outcomes. Compare the local model SHA-256 and its manifest. The run **does not load or fit the estimator**; its saved probability vectors are the source of truth.

*Never change this notebook's threshold, baseline definitions or sample-selection rule based on interim scores. Changes, if any, must be versioned and disclosed prospectively.*

In [ ]:
def sha256_file(path):
    return audit_sha256_file(Path(path))

def read_json(path):
    with open(path,"r",encoding="utf-8-sig") as f:
        obj=json.load(f)
    if not isinstance(obj,dict):
        raise ValueError(f"Expected JSON object: {path.name}")
    return obj

model_file = MODEL_DIR / "es_daily_lr_deployment_v0_1.joblib"
manifest_file = MODEL_DIR / "es_daily_lr_deployment_v0_1_manifest.json"
metadata_file = MODEL_DIR / "es_daily_lr_deployment_v0_1_metadata.json"
MODEL_AUDIT = audit_model(PROJECT_ROOT)
LOCAL_MODEL_VERIFIED = MODEL_AUDIT["verified"]
print("LOCAL FROZEN DEPLOYMENT:", "VERIFIED" if LOCAL_MODEL_VERIFIED else "NOT VERIFIED")
print("Model issues:", MODEL_AUDIT["issues"])
if not LOCAL_MODEL_VERIFIED:
    warnings.warn("Original local model and metadata not completely verified. Do not publish final model-provenance claims from this run.")
print("pandas:",pd.__version__,"numpy:",np.__version__)
print("UTC analysis time:",datetime.now(timezone.utc).isoformat())

## 02 · Input contracts (private CSV templates)

**Arena official settlements** — `arena_official_settlements.csv`: 1 row per challenge, with official `official_label` (`bearish/neutral/bullish`), `settlement_date` (YYYY-MM-DD), `challenge_id`, `asset` = `ES`, `contract_symbol` (typically `ESZ6`), `source_url` (official Arena challenge URL). Optional: `settlement_return_pct` in **percent units** (e.g. `-0.438` means −0.438%), `official_deadline_utc`, `official_settled_at_utc`, `arena_score`, `verified_by`, `verified_at_utc`. **No official result = no settled metric.** Do not use an empty `Outcome —` as an outcome.

**Peer records** — `arena_peer_forecasts.csv`: one row per `challenge_id` × `agent_id`, with exact three probability columns `p_bearish,p_neutral,p_bullish`, `submitted_at_utc`, `source_url` and optionally `agent_name`, `counts_for_score`. Include only publicly accessible and permitted records, never assume leaderboard summaries represent matched predictions.

**Training counts** — `training_class_counts.csv`: exactly three rows with `class_label,count`, obtained from *training-period only* observations and documented in `source_description`; absent by default because the training labels were not supplied with this notebook. We never estimate historical priors from the evaluation outcomes.

**Calendar** — `evaluation_calendar.csv`: optionally record `settlement_date,status,notes` for closed, missed or unoffered challenges, so non-submissions are visible. Accepted but unresolved records are shown separately.

**Recovered first submission:** A JSON record in `legacy_submissions/` preserves the original HTTP 201 response, prediction ID, exact probabilities and public challenge source. The record must be explicitly marked as a transcription of contemporaneous console evidence, **never** as the original receipt. A separate official-settlements row (with verifiable Arena source URL) supplies its actual bullish outcome.

Blank templates do not constitute observations. Native forecast/receipt JSON schemas may evolve; an unrecognized format is surfaced for adapter review, not silently guessed.

**Git safety prerequisite:** before enabling `WRITE_MISSING_TEMPLATES`, ensure `.gitignore` contains `data/processed/es_arena_research_v0_1/` (or an equivalent broader ignore rule). The template-creation cell checks Git exclusion and refuses to write otherwise.


**Seed row rule:** If you have independently verified the October 20 official result, you may place the separately supplied one-row, sourced CSV at `arena_official_settlements.csv` before enabling `WRITE_MISSING_TEMPLATES=True`; the creation step will preserve it. Never replace an existing, populated settlements CSV with a one-row seed.


In [ ]:
TEMPLATES = {
    OFFICIAL_CSV: ["challenge_id","settlement_date","asset","contract_symbol","official_label","settlement_return_pct","official_deadline_utc","official_settled_at_utc","arena_score","source_url","verified_by","verified_at_utc"],
    PEERS_CSV: ["challenge_id","agent_id","agent_name","p_bearish","p_neutral","p_bullish","submitted_at_utc","counts_for_score","source_url"],
    TRAIN_COUNTS_CSV: ["class_label","count","source_description"],
    SCHEDULE_CSV: ["settlement_date","status","notes"]
}
if WRITE_MISSING_TEMPLATES:
    import subprocess
    assert (PROJECT_ROOT / ".gitignore").exists(), "Review .gitignore first"
    for path in TEMPLATES:
        check=subprocess.run(["git","-C",str(PROJECT_ROOT),"check-ignore","-q",str(path)],capture_output=True)
        if check.returncode!=0:
            raise RuntimeError(f"STOP: {path} is not Git-ignored. Add 'data/processed/es_arena_research_v0_1/' to .gitignore before creating input templates")
    RESEARCH_DIR.mkdir(parents=True,exist_ok=True)
    for path, cols in TEMPLATES.items():
        if path.exists():
            print("PRESERVED EXISTING:", path.name)
        else:
            with path.open("x",encoding="utf-8",newline="") as f:
                f.write(",".join(cols)+"\n")
            print("CREATED EMPTY TEMPLATE:",path)
else:
    for path in TEMPLATES:
        print("PRESENT" if path.exists() else "NOT PROVIDED",path.name)

def load_optional_csv(path):
    if not path.exists():
        return pd.DataFrame(columns=TEMPLATES[path])
    df=pd.read_csv(path, dtype=str,keep_default_na=False)
    missing=set(TEMPLATES[path]).difference(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing required headers: {sorted(missing)}")
    return df

arena_raw=load_optional_csv(OFFICIAL_CSV)
peer_raw=load_optional_csv(PEERS_CSV)
train_raw=load_optional_csv(TRAIN_COUNTS_CSV)
calendar_raw=load_optional_csv(SCHEDULE_CSV)
print(f"Human-verified official rows: {len(arena_raw)} | peer rows: {len(peer_raw)} | calendar rows: {len(calendar_raw)}")

## 03 · Read original native forecasts — canonical dates required

The original forecast writer records `target_session_date` and `last_eligible_session` (not `target_date` / `last_session`). Native records **must** contain both. Missing or contradictory fields are audit failures, never an implicit pass. Native forecast probabilities are read from the saved immutable JSON; the estimator is **never rerun**.

The supporting tool independently checks local model bytes, manifest and metadata, frozen-model source SHA, snapshot bytes and manifest, reviewed-audit hash, completed source session and model inputs. It records issues per challenge. An invalid native record remains visible in the audit summary; it is not silently reclassified as a successful accepted forecast.

In [ ]:
def at_path(obj, path):
    cur=obj
    for key in path.split("."):
        if not isinstance(cur,dict) or key not in cur:
            return None
        cur=cur[key]
    return cur

def candidates_at_paths(obj, paths):
    return [(path,at_path(obj,path)) for path in paths if at_path(obj,path) is not None]

def unique_value(obj, paths, label):
    values=candidates_at_paths(obj,paths)
    if not values:return None
    if len({str(v).strip() for _,v in values})>1:
        raise ValueError(f"Conflicting {label} fields")
    return values[0][1]

PROB_PATHS = ["probabilities", "class_probabilities", "forecast.probabilities",
              "prediction.probabilities", "payload.probabilities", "prediction_payload.probabilities",
              "forecast", "prediction", "payload", "prediction_payload", ""]

def parse_probabilities(obj):
    found=[]
    for path in PROB_PATHS:
        node=at_path(obj,path) if path else obj
        if not isinstance(node,dict):continue
        for keys in [tuple(f"p_{c}" for c in CLASSES),CLASSES]:
            if all(k in node for k in keys):
                vec=np.array([float(node[k]) for k in keys],dtype=float)
                found.append((path or "root",vec))
    if not found:raise ValueError("No explicit three-class probabilities")
    p=found[0][1]
    if any(not np.allclose(p,x[1],rtol=0,atol=1e-10) for x in found[1:]):
        raise ValueError("Contradictory probability vectors")
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any() or abs(p.sum()-1)>1e-10:
        raise ValueError("Invalid three-class probabilities")
    return p

FORECAST_NAME=re.compile(r"^forecast_(\d{8})_([0-9a-fA-F-]{36})\.json$")

PROVENANCE_AUDIT = audit_repository(PROJECT_ROOT)
provenance_df = pd.DataFrame(PROVENANCE_AUDIT["records"])
print("LOCAL PROVENANCE SUMMARY:",PROVENANCE_AUDIT["summary"])
if not provenance_df.empty:
    display(provenance_df[["challenge_id","target_session_date","last_eligible_session",
      "model_hash_verified","snapshot_hash_verified","receipt_forecast_hash_verified",
      "receipt_server_accepted","deadline_verified","provenance_status"]])
    problems=provenance_df.loc[provenance_df.provenance_status.isin(["FAIL","INCOMPLETE"])]
    if not problems.empty:
        print("UNRESOLVED NATIVE EVIDENCE — must investigate; never silently upgrade:")
        display(problems[["challenge_id","provenance_status","issues","warnings"]])

AUDIT_BY_CHALLENGE = {r["challenge_id"]:r for r in PROVENANCE_AUDIT["records"] if r["evidence_type"]=="NATIVE"}

def load_forecast_file(path):
    obj=read_json(path)
    match=FORECAST_NAME.fullmatch(path.name)
    if not match:
        raise ValueError("Native filename invalid")
    date_iso=f"{match[1][:4]}-{match[1][4:6]}-{match[1][6:8]}"
    cid=match[2].lower()
    if obj.get("challenge_id")!=cid:
        raise ValueError("Canonical challenge_id missing/mismatch")
    if obj.get("target_session_date")!=date_iso:
        raise ValueError("Canonical target_session_date missing/mismatch")
    last=obj.get("last_eligible_session")
    if not isinstance(last,str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}",last) or not last<date_iso:
        raise ValueError("Canonical last_eligible_session missing/invalid/non-prior")
    contract=obj.get("contract_symbol")
    if not isinstance(contract,str) or not contract:
        raise ValueError("Canonical contract_symbol missing")
    p=parse_probabilities(obj)
    audit=AUDIT_BY_CHALLENGE.get(cid)
    if audit is None:
        raise ValueError("No associated provenance audit record")
    return dict(challenge_id=cid,settlement_date=date_iso,contract_symbol=contract,
                p_bearish=p[0],p_neutral=p[1],p_bullish=p[2],
                predicted_label=CLASSES[int(np.argmax(p))],confidence=float(p.max()),
                forecast_created_at_utc=str(obj.get("prepared_at_utc") or ""),
                last_session=last,forecast_file=path.name,forecast_sha256=sha256_file(path),
                original_model_sha_verified=bool(audit["model_hash_verified"]),
                model_manifest_verified=bool(audit["model_manifest_verified"]),
                source_hash_verified=bool(audit["snapshot_hash_verified"]),
                provenance_status=audit["provenance_status"],
                strict_native_provenance=audit["provenance_status"]=="VERIFIED_LOCAL_CHAIN",
                artifact_integrity_verified=bool(audit["artifact_integrity_verified"]),
                submitted_at_utc_verified=bool(audit["deadline_verified"]),
                receipt_server_accepted=bool(audit["receipt_server_accepted"]))

forecast_paths=sorted(FORECAST_DIR.glob("forecast_*.json")) if FORECAST_DIR.is_dir() else []
forecast_rows=[]; forecast_errors=[]
for path in forecast_paths:
    try:forecast_rows.append(load_forecast_file(path))
    except Exception as exc:forecast_errors.append((path.name,str(exc)))
forecasts=pd.DataFrame(forecast_rows)
print("Native forecasts found:",len(forecast_paths),"| canonical parsed:",len(forecasts),
      "| rejected records:",len(forecast_errors))
if forecast_errors:
    display(pd.DataFrame(forecast_errors,columns=["file","problem"]))
    warnings.warn("Invalid native forecast files exist. Final research publication blocked; preserve them for investigation.")
if not forecasts.empty:
    assert not forecasts.challenge_id.duplicated().any(),"Duplicate native challenge: STOP"
    display(forecasts[["settlement_date","challenge_id","predicted_label","confidence",
                       "original_model_sha_verified","provenance_status"]].sort_values("settlement_date"))

## 04 · Original scored receipts, client/server time distinction and recovered October 20 evidence

For native forecasts, an accepted receipt requires HTTP 201, `counts_for_score=True`, matching challenge and prediction identifiers, and the receipt's forecast-SHA matching the **original** saved forecast bytes. Failed or missing acceptance is not silently counted as accepted.

**Important:** `response.created_at` without timezone is kept raw, **not** converted to UTC. `receipt.recorded_at_utc` is an offset-aware *local collection timestamp*; it can be compared to a published deadline as ancillary evidence, **not** independent verification of server acceptance time. Even if the clocks appear synchronized, the strict verifier remains `INCOMPLETE` until independently suitable server evidence exists.

October 20 remains an explicit, lower-provenance legacy exception and is shown separately in every sensitivity summary.

In [ ]:
def first_nested(obj, paths):
    for path in paths:
        value=at_path(obj,path)
        if value is not None:return value
    return None

def parse_utc(v):
    if v is None or (isinstance(v,str) and not v.strip()):return pd.NaT
    # Never pretend timezone-naive server text is UTC.
    if isinstance(v,str) and re.search(r"(?:Z|[+-]\d{2}:\d{2})$",v.strip()) is None:
        return pd.NaT
    value=pd.to_datetime(v,utc=True,errors="coerce")
    return pd.NaT if isinstance(value,pd.DatetimeIndex) else value

def bool_is_true(v):
    return v is True or (isinstance(v,str) and v.strip().lower()=="true")

RECEIPT_NAME=re.compile(r"^receipt_([0-9a-fA-F-]{36})\.json$")
receipt_paths=sorted(RECEIPT_DIR.glob("receipt_*.json")) if RECEIPT_DIR.is_dir() else []
receipt_rows=[]
receipt_errors=[]
for path in receipt_paths:
    match=RECEIPT_NAME.fullmatch(path.name)
    if not match:
        receipt_errors.append((path.name,"Invalid receipt filename"));continue
    cid=match[1].lower()
    audit=AUDIT_BY_CHALLENGE.get(cid)
    if audit is None:
        receipt_errors.append((path.name,"Orphan receipt without canonical native forecast"));continue
    receipt_rows.append(dict(
        challenge_id=cid,receipt_file=path.name,receipt_sha256=sha256_file(path),
        receipt_http_status=audit["receipt_http_status"],
        receipt_counts_for_score=audit["receipt_counts_for_score"],
        accepted_by_receipt=bool(audit["accepted_by_receipt"]),
        submitted_at_utc=audit["submitted_at_utc"],
        receipt_forecast_hash_verified=bool(audit["receipt_forecast_hash_verified"]),
        receipt_deadline_verified=bool(audit["deadline_verified"]),
        attempt_present=bool(audit["attempt_present"])))
# Supplemental, READ-ONLY timestamp disclosure. The client's timezone-aware
# recorded_at_utc cannot repair a server created_at without timezone.
for rr in receipt_rows:
    rp = RECEIPT_DIR / rr["receipt_file"]
    try:
        original = read_json(rp)
        server = original.get("response") if isinstance(original.get("response"), dict) else {}
        rr["client_recorded_at_utc"] = str(original.get("recorded_at_utc", ""))
        rr["server_created_at_raw"] = str(server.get("created_at", ""))
        rr["client_recorded_time_aware"] = pd.notna(parse_utc(rr["client_recorded_at_utc"]))
        rr["server_created_at_aware"] = pd.notna(parse_utc(rr["server_created_at_raw"]))
    except (OSError, ValueError):
        rr["client_recorded_at_utc"] = ""
        rr["server_created_at_raw"] = ""
        rr["client_recorded_time_aware"] = False
        rr["server_created_at_aware"] = False

receipts=pd.DataFrame(receipt_rows)
if receipt_errors:
    display(pd.DataFrame(receipt_errors,columns=["receipt_file","problem"]))
print("Receipt files:",len(receipt_paths),"| strict scored + hash-linked:",
      int(receipts.accepted_by_receipt.sum()) if not receipts.empty else 0,
      "| missing/orphan/invalid:",len(receipt_errors))
if not receipts.empty:
    print("RECEIPT TIME DISCLOSURE — local recording time does not certify server timezone:")
    display(receipts[["challenge_id","client_recorded_at_utc","server_created_at_raw",
                      "client_recorded_time_aware","server_created_at_aware"]])
if not receipts.empty:
    assert not receipts.challenge_id.duplicated().any(),"Duplicate receipt challenges: STOP"
    display(receipts[["challenge_id","accepted_by_receipt",
                      "receipt_forecast_hash_verified","receipt_deadline_verified","attempt_present"]])

# IMPORTANT: recover independently verified first-notebook submissions without fabricating
# a modern immutable forecast or a modern receipt. The original evidence file is never edited.
from urllib.parse import urlparse
import uuid

def load_recovered_legacy(path):
    obj=read_json(path)
    if obj.get("evidence_kind")!="recovered_original_cli_transcription_v1":
        raise ValueError("Legacy evidence must be explicitly marked as reconstructed from original console output")
    cid=str(obj.get("challenge_id", "")).lower()
    if not cid or str(uuid.UUID(cid))!=cid:
        raise ValueError("Legacy challenge UUID malformed")
    result=obj.get("server_response")
    if not isinstance(result,dict): raise ValueError("Missing preserved original server response")
    if str(result.get("challenge_id", "")).lower()!=cid: raise ValueError("Server/challenge ID conflict")
    if str(result.get("prediction_id", "")).strip()=="": raise ValueError("No server prediction ID")
    if str(obj.get("http_status"))!="201" or result.get("counts_for_score") is not True:
        raise ValueError("Legacy record not confirmed accepted with HTTP 201 and scored=True")
    date=str(obj.get("settlement_date", ""))
    if not re.fullmatch(r"20\d{2}-\d{2}-\d{2}",date): raise ValueError("Bad legacy date")
    if str(obj.get("contract_symbol", ""))!=EXPECTED_CONTRACT: raise ValueError("Unexpected legacy contract")
    url=str(obj.get("official_challenge_url", ""))
    parsed=urlparse(url)
    if parsed.scheme!="https" or parsed.hostname not in {"headlinearena.com","www.headlinearena.com"} or cid not in parsed.path:
        raise ValueError("Recovered record needs exact Arena public challenge URL")
    if not str(obj.get("source_note", "")).strip(): raise ValueError("Explicit legacy provenance note required")
    probabilities=parse_probabilities(result)
    direction=CLASSES[int(np.argmax(probabilities))]
    if str(result.get("direction"))!=direction: raise ValueError("Direction inconsistent with recorded probabilities")
    if not np.isclose(float(result.get("confidence")),probabilities.max(),atol=1e-10,rtol=0):
        raise ValueError("Legacy confidence differs from probabilities")
    # The API server's raw timestamp has no UTC offset. NEVER invent one.
    raw_time=str(result.get("created_at", ""))
    if not raw_time: raise ValueError("Missing original raw server creation timestamp")
    return dict(challenge_id=cid,settlement_date=date,contract_symbol=EXPECTED_CONTRACT,
                p_bearish=float(probabilities[0]),p_neutral=float(probabilities[1]),p_bullish=float(probabilities[2]),
                predicted_label=direction,confidence=float(probabilities.max()),
                forecast_created_at_utc="",last_session="",forecast_file="",forecast_sha256="",
                receipt_file="",receipt_sha256="",receipt_http_status="201",receipt_counts_for_score="True",
                accepted_by_receipt=False,accepted_by_external_evidence=True,
                submitted_at_utc="",legacy_server_created_at_raw=raw_time,
                evidence_type="RECOVERED_CLI_201_PLUS_OFFICIAL_PAGE",evidence_file=path.name,
                legacy_file_sha256=sha256_file(path),native_artifacts_present=False,
                original_model_sha_verified=False)

legacy_paths=sorted(LEGACY_DIR.glob("legacy_*.json")) if LEGACY_DIR.is_dir() else []
legacy_rows=[];legacy_errors=[]
for path in legacy_paths:
    try: legacy_rows.append(load_recovered_legacy(path))
    except Exception as exc: legacy_errors.append((path.name,str(exc)))
if legacy_errors:
    display(pd.DataFrame(legacy_errors,columns=["legacy_file","error"]))
    raise ValueError("Legacy evidence failed validation — STOP; correct it before continuing")
if legacy_rows:
    print("RECOVERED ORIGINAL CONSOLE EVIDENCE (NOT ORIGINAL JSON OR RECEIPT):",len(legacy_rows))
    display(pd.DataFrame(legacy_rows)[["settlement_date","challenge_id","predicted_label","confidence","evidence_type"]])
else:
    print("NO LEGACY FIRST-NOTEBOOK EVIDENCE FOUND — cohort may be missing October 20")


# Preserve accepted records as historical evidence, independently of whether
# their complete model/source chain can currently be verified.
native = forecasts.merge(receipts,on="challenge_id",how="left",validate="one_to_one") if not forecasts.empty else forecasts.copy()
if not native.empty:
    if "accepted_by_receipt" not in native:native["accepted_by_receipt"]=False
    native["accepted_by_receipt"]=native["accepted_by_receipt"].fillna(False).astype(bool)
    if "receipt_server_accepted" not in native:native["receipt_server_accepted"]=False
    native["receipt_server_accepted"]=native["receipt_server_accepted"].fillna(False).astype(bool)
    native["accepted_by_external_evidence"]=False
    native["evidence_type"]="NATIVE_FORECAST_PLUS_RECEIPT"
    native["native_artifacts_present"]=True
    # NO unconditional original_model_sha_verified=True.
    native["original_model_sha_verified"]=native["original_model_sha_verified"].fillna(False).astype(bool)
    native["strict_native_provenance"]=native["strict_native_provenance"].fillna(False).astype(bool)
    native["legacy_server_created_at_raw"]=""
    native["evidence_file"]=""
    native["legacy_file_sha256"]=""

if legacy_rows and not native.empty:
    overlap=set(native.challenge_id) & {r["challenge_id"] for r in legacy_rows}
    if overlap:raise ValueError(f"Native and legacy evidence overlap: {sorted(overlap)}")
# If a native forecast is malformed but there is an authentic server-scored receipt,
# retain a visibly quarantined placeholder in the chronological cohort. Never
# silently replace its opportunity with a later favorable forecast.
quarantined=[]
for rec in PROVENANCE_AUDIT["records"]:
    if rec["evidence_type"]!="NATIVE" or not rec["receipt_server_accepted"]:
        continue
    if not native.empty and rec["challenge_id"] in set(native.challenge_id):
        continue
    match=FORECAST_NAME.fullmatch(rec["forecast_file"] or "")
    if match is None:
        print("UNRANKABLE SCORED RECORD: invalid native filename; manual cohort reconciliation required",rec["challenge_id"])
        continue
    name_date=f"{match[1][:4]}-{match[1][4:6]}-{match[1][6:8]}"
    quarantined.append(dict(challenge_id=rec["challenge_id"],settlement_date=name_date,
       contract_symbol=rec["contract_symbol"],predicted_label="",confidence=np.nan,
       p_bearish=np.nan,p_neutral=np.nan,p_bullish=np.nan,
       accepted_by_receipt=False,receipt_server_accepted=True,
       accepted_by_external_evidence=False,strict_native_provenance=False,
       original_model_sha_verified=False,native_artifacts_present=True,
       last_session=rec.get("last_eligible_session", ""),
       evidence_type="QUARANTINED_NATIVE_SERVER_ACCEPTED",
       provenance_status="FAIL",forecast_file=rec["forecast_file"]))
if quarantined:
    print("QUARANTINED SERVER-ACCEPTED CHALLENGES (cohort positions reserved; excluded from metrics):",len(quarantined))
    display(pd.DataFrame(quarantined)[["challenge_id","settlement_date","evidence_type"]])
frames=[x for x in (native,pd.DataFrame(quarantined),pd.DataFrame(legacy_rows)) if not x.empty]
ledger=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame(
    columns=["challenge_id","settlement_date","predicted_label","confidence",
             "accepted_by_receipt","accepted_by_external_evidence"])
if not ledger.empty:
    assert not ledger.challenge_id.duplicated().any(),"Duplicate challenge evidence"
    for key in ("accepted_by_receipt","accepted_by_external_evidence"):
        ledger[key]=ledger[key].fillna(False).astype(bool)
    ledger["receipt_server_accepted"]=ledger.get("receipt_server_accepted",pd.Series(index=ledger.index,dtype=bool)).fillna(False).astype(bool)
    ledger["accepted_with_evidence"]=ledger.accepted_by_receipt | ledger.accepted_by_external_evidence | ledger.receipt_server_accepted
    ledger["strict_native_provenance"]=ledger.get("strict_native_provenance",pd.Series(index=ledger.index,dtype=bool)).fillna(False).astype(bool)
    ledger=ledger.sort_values(["settlement_date","challenge_id"]).reset_index(drop=True)
    ledger["accepted_rank"]=np.nan
    accepted_pos=ledger.index[ledger.accepted_with_evidence & (ledger.settlement_date>=COHORT_START)]
    ledger.loc[accepted_pos,"accepted_rank"]=np.arange(1,len(accepted_pos)+1)
    ledger["in_first_30_accepted"]=ledger.accepted_rank.between(1,COHORT_TARGET_N)
    ledger["source_date_before_target"]=ledger.apply(
        lambda r: (bool(r.accepted_by_external_evidence) and not str(r.get("last_session","") or "").strip())
        or (isinstance(r.get("last_session"),str) and bool(re.fullmatch(r"\d{4}-\d{2}-\d{2}",r.last_session))
            and r.last_session<r.settlement_date),axis=1)
    if not ledger.source_date_before_target.all():
        warnings.warn("Some accepted records have an invalid/missing source date. They remain in the cohort but are excluded from qualified metrics; FINAL publication blocked.")
        display(ledger.loc[~ledger.source_date_before_target,["challenge_id","settlement_date","last_session","provenance_status"]])
else:
    ledger["accepted_with_evidence"]=pd.Series(dtype=bool)
    ledger["in_first_30_accepted"]=pd.Series(dtype=bool)
    ledger["strict_native_provenance"]=pd.Series(dtype=bool)
print("Confirmed native accepted receipts:",int(native.accepted_by_receipt.sum()) if not native.empty else 0)
print("Accepted recovered legacy submissions:",len(legacy_rows))
print("Accepted cohort ledger entries:",int(ledger.accepted_with_evidence.sum()) if not ledger.empty else 0)
if ORIGINAL_FIRST_CHALLENGE_ID not in (set(ledger.challenge_id) if not ledger.empty else set()):
    print("COHORT WARNING: first 2026-10-20 record missing; do not silently substitute challenge 31")
if not ledger.empty:
    display(ledger[["settlement_date","challenge_id","accepted_by_receipt",
      "accepted_by_external_evidence","original_model_sha_verified",
      "strict_native_provenance","accepted_rank","in_first_30_accepted"]])

## 05 · Official outcomes: separate descriptive scoring from strict provenance certification

**Main descriptive sample** = first-30 accepted opportunity, official matched ES settlement, saved original three-class probabilities, and adequate acceptance/integrity support. For native forecasts, the existing verifier must establish actual local model/source integrity, scored HTTP-201 acceptance and matching receipt/forecast SHA, **with no `FAIL` or impossible source-date ordering**. If independent server UTC time is unknown, include the record in *descriptive* accuracy/Brier but label it `SERVER_UTC_UNRESOLVED`. October 20 is accepted **only** through its explicitly recovered evidence; this is disclosed and examined in a separate sensitivity analysis.

**Strict-provenance sensitivity** requires the original verifier's full `VERIFIED_LOCAL_CHAIN` status. It can have zero observations. Its absence must not cause the main evaluation to display a misleading 100%-accuracy result from only one legacy observation.

Other caveats: an Arena 201/scored response is evidence of scored acceptance, not a signed server-time attestation. `recorded_at_utc` reflects local client observation. An independent server on-time comparison cannot be made using `response.created_at` until the timezone semantics are verified externally. Preserve every failed and missing challenge in the fixed cohort.

In [ ]:
def validate_official(df):
    if df.empty: return df.copy()
    d=df.copy()
    assert d.challenge_id.ne("").all() and not d.challenge_id.duplicated().any(), "Blank or duplicate challenge IDs in official CSV"
    assert d.official_label.isin(CLASSES).all(),"Invalid official labels"
    assert d.asset.eq("ES").all(),"Official CSV must be ES-only"
    assert d.settlement_date.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all(),"Bad settlement dates"
    assert d.source_url.ne("").all(),"Official source URL required"
    from urllib.parse import urlparse
    assert d.apply(lambda r: (urlparse(r.source_url).scheme=="https") and
                   (urlparse(r.source_url).hostname in {"headlinearena.com","www.headlinearena.com"}) and
                   (r.challenge_id.lower() in urlparse(r.source_url).path.lower()),axis=1).all(),"Use an HTTPS Arena source URL for this exact challenge"
    assert d.contract_symbol.ne("").all(),"Contract required to verify same underlying"
    d["challenge_id"]=d.challenge_id.str.lower().str.strip()
    d["settlement_return_pct"]=pd.to_numeric(d.settlement_return_pct.replace("",np.nan),errors="coerce")
    d["arena_score"]=pd.to_numeric(d.arena_score.replace("",np.nan),errors="coerce")
    return d

arena=validate_official(arena_raw)
if ledger.empty:
    full=ledger.copy()
else:
    full=ledger.merge(arena,on="challenge_id",how="left",suffixes=("","_official"),validate="one_to_one") if not arena.empty else ledger.copy()
    if "official_label" not in full: full["official_label"]=np.nan
    if "settlement_date_official" in full:
        conflict=full.settlement_date_official.notna() & (full.settlement_date_official!=full.settlement_date)
        if conflict.any(): raise ValueError("Challenge date in official CSV mismatches forecast: STOP")
    if "contract_symbol_official" in full:
        conflict=full.contract_symbol_official.notna() & (full.contract_symbol_official!=full.contract_symbol)
        if conflict.any(): raise ValueError("Official contract differs: STOP")
    full["settled_confirmed"]=full.official_label.isin(CLASSES)

    # The descriptive sample is defined independently of STRONGER timestamp certification.
    # An actual hash contradiction (FAIL), missing scored receipt or source integrity failure
    # is still excluded; timezone-uncertain valid receipts stay visible as descriptive evidence.
    legacy=full.accepted_by_external_evidence.fillna(False).astype(bool)
    native_receipt=full.accepted_by_receipt.fillna(False).astype(bool)
    integrity=full.get("artifact_integrity_verified",pd.Series(False,index=full.index)).fillna(False).astype(bool)
    provenance_ok=~full.get("provenance_status",pd.Series("INCOMPLETE",index=full.index)).eq("FAIL")
    source_ok=full.get("source_date_before_target",pd.Series(False,index=full.index)).fillna(False).astype(bool)
    prob=full[[f"p_{c}" for c in CLASSES]].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
    probability_ok=pd.Series(np.isfinite(prob).all(axis=1) & (prob >= 0).all(axis=1) &
                             (prob <= 1).all(axis=1) & np.isclose(prob.sum(axis=1),1,rtol=0,atol=1e-10),index=full.index)
    native_descriptive=native_receipt & integrity & provenance_ok & source_ok & probability_ok
    legacy_descriptive=legacy & probability_ok

    full["eligible_for_descriptive_metrics"]=(full.in_first_30_accepted.fillna(False) &
       full.settled_confirmed & (native_descriptive | legacy_descriptive))
    full["eligible_for_strict_provenance_metrics"]=(full.in_first_30_accepted.fillna(False) &
       full.settled_confirmed & native_descriptive & full.strict_native_provenance.fillna(False).astype(bool))
    # Maintain a backward-compatible alias for existing visualizations and peer comparison code.
    full["eligible_for_metrics"]=full.eligible_for_descriptive_metrics
    full["excluded_from_descriptive_metrics"]=(full.in_first_30_accepted.fillna(False) &
       full.settled_confirmed & ~full.eligible_for_descriptive_metrics)
    full["excluded_due_to_native_provenance"]=full.excluded_from_descriptive_metrics

    # This is not an independent acceptance-time certification. Compare only aware timestamps.
    full["submission_deadline_status"]="UNKNOWN_SERVER_UTC"
    if "official_deadline_utc" in full:
        all_status=[]
        for _, r in full.iterrows():
            observed=parse_utc(r.get("submitted_at_utc"))
            deadline=parse_utc(r.get("official_deadline_utc"))
            if pd.notna(observed) and pd.notna(deadline):
                all_status.append("VERIFIED_SERVER_ON_TIME" if observed<=deadline else "SERVER_LATE")
            elif bool(r.get("accepted_by_external_evidence",False)):
                all_status.append("LEGACY_SCORED_UNZONED_TIME")
            elif bool(r.get("accepted_by_receipt",False)):
                all_status.append("SCORED_SERVER_UTC_UNRESOLVED")
            else:
                all_status.append("UNKNOWN_SERVER_UTC")
        full["submission_deadline_status"]=all_status
        if full.submission_deadline_status.eq("SERVER_LATE").any():
            raise ValueError("Confirmed server-after-deadline submission: STOP; investigate before reporting metrics")
    else:
        full.loc[legacy,"submission_deadline_status"]="LEGACY_SCORED_UNZONED_TIME"
        full.loc[native_receipt,"submission_deadline_status"]="SCORED_SERVER_UTC_UNRESOLVED"

    full["client_receipt_recorded_at_utc"]=""
    full["client_recorded_predeadline"]="UNKNOWN"
    if not receipts.empty and "client_recorded_at_utc" in receipts:
        time_lookup=receipts.set_index("challenge_id")["client_recorded_at_utc"].to_dict()
        full["client_receipt_recorded_at_utc"]=full.challenge_id.map(time_lookup).fillna("")
        if "official_deadline_utc" in full:
            for index, row in full.iterrows():
                t=parse_utc(row.client_receipt_recorded_at_utc)
                deadline=parse_utc(row.official_deadline_utc)
                if pd.notna(t) and pd.notna(deadline):
                    full.at[index,"client_recorded_predeadline"]="YES_CLIENT_OBSERVED" if t<=deadline else "NO_CLIENT_OBSERVED"

# Venue deadlines from the official CSV remain separate from the forecast's declared deadline.
if not full.empty and "official_deadline_utc" in full:
    official_time_conflicts=[]
    for rec in full.to_dict("records"):
        local=AUDIT_BY_CHALLENGE.get(rec["challenge_id"])
        o=parse_utc(rec.get("official_deadline_utc"))
        l=parse_utc(local.get("deadline_utc")) if local else pd.NaT
        if pd.notna(o) and pd.notna(l) and o != l:
            official_time_conflicts.append(rec["challenge_id"])
    if official_time_conflicts:
        raise ValueError("Official Arena and local forecast deadlines conflict: " + str(official_time_conflicts))

print("Official locally recorded settlement rows:",len(arena),"— may lag live Arena")
print("Accepted first-30, officially settled, descriptively qualified:",
      int(full.eligible_for_descriptive_metrics.sum()) if not full.empty else 0)
print("Strict UTC-certified native subset:",
      int(full.eligible_for_strict_provenance_metrics.sum()) if not full.empty else 0)
print("These are DIFFERENT analysis populations; unresolved server UTC does NOT certify an on-time server timestamp.")
if not full.empty:
    display(full[["settlement_date","challenge_id","predicted_label","official_label",
                  "accepted_by_receipt","provenance_status","artifact_integrity_verified",
                  "eligible_for_descriptive_metrics","eligible_for_strict_provenance_metrics",
                  "submission_deadline_status","client_recorded_predeadline"]])
    if full.excluded_from_descriptive_metrics.any():
        print("QUARANTINED settled first-30 opportunities — not replaced; investigate:")
        display(full.loc[full.excluded_from_descriptive_metrics,
              ["settlement_date","challenge_id","provenance_status","accepted_by_receipt"]])
if not full.empty and full.eligible_for_descriptive_metrics.sum()<COHORT_TARGET_N:
    print("PRELIMINARY: not all first-30 accepted opportunities have official outcomes and adequate descriptive evidence.")

## 06 · Metric definitions and self-tests

**Primary:** accuracy of the maximum-probability class; *unscaled* three-class Brier score `mean(sum_k (p_k - 1[y=k])²)`, lower is better (range 0–2). This may differ numerically from an Arena-normalized score. **Secondary:** log loss (epsilon = `1e-15`), confidence, per-class precision/recall, macro-F1, and multiclass confusion matrix. Use official Arena class, not locally inferred class.

Self-tests below use **obviously artificial arrays** to catch arithmetic mistakes. They are never presented as model performance.

In [ ]:
def arrays_from_rows(frame):
    p=frame[[f"p_{c}" for c in CLASSES]].to_numpy(dtype=float)
    labels=frame.official_label.astype(str).to_numpy()
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any() or not np.allclose(p.sum(axis=1),1,atol=1e-6):
        raise ValueError("Invalid model probability vectors")
    if not set(labels).issubset(CLASSES): raise ValueError("Invalid official labels")
    return p,np.array([CLASS_TO_I[x] for x in labels],dtype=int)

def score_probabilities(p,y):
    p=np.asarray(p,dtype=float)
    y=np.asarray(y,dtype=int)
    assert p.ndim==2 and p.shape[1]==3 and len(y)==len(p) and len(y)>0
    assert np.allclose(p.sum(axis=1),1,atol=1e-6)
    one=np.eye(3)[y]
    pred=p.argmax(axis=1)
    return dict(n=len(y),accuracy=float(np.mean(pred==y)),
        avg_confidence=float(p.max(axis=1).mean()),
        brier=float(np.sum((p-one)**2,axis=1).mean()),
        log_loss=float(-np.log(np.clip(p[np.arange(len(y)),y],1e-15,1)).mean()))

# Unit tests: perfect, uniform, degenerate and general three-class distributions.
P=np.eye(3)
Y=np.array([0,1,2])
assert score_probabilities(P,Y)["brier"]==0 and score_probabilities(P,Y)["accuracy"]==1
U=np.full((3,3),1/3)
assert np.isclose(score_probabilities(U,Y)["brier"],2/3)
assert np.isclose(score_probabilities(U,Y)["log_loss"],np.log(3))
assert np.isclose(score_probabilities(np.array([[1.,0.,0.]]),np.array([2]))["brier"],2)
print("METRIC UNIT TESTS: PASS (test examples are NOT research results)")

## 07 · Main descriptive forecasting results — available accepted and officially settled challenges

**Primary display = descriptive forecasting performance**, not an independent timing certification. Only original recorded probability vectors with official outcomes and sufficiently consistent scored acceptance/artifact evidence enter this table. Native records whose *only* unresolved issue is independent server UTC acceptance time are included with a prominent warning. Other invalid/incomplete evidence is excluded and listed rather than silently replaced. The October 20 recovered record is explicitly lower-provenance; an exclusion sensitivity is shown below.

Do **not** interpret a single observation or early cohort as definitive model skill. No model is retrained. Later phases are not included.

In [ ]:
eval_df = full.loc[full.eligible_for_descriptive_metrics].copy() if not full.empty else pd.DataFrame()
strict_eval_df = full.loc[full.eligible_for_strict_provenance_metrics].copy() if not full.empty else pd.DataFrame()
print("MAIN DESCRIPTIVE SAMPLE:",len(eval_df),"| STRICT UTC-CERTIFIED NATIVE SAMPLE:",len(strict_eval_df))
if not eval_df.empty and (eval_df.submission_deadline_status != "VERIFIED_SERVER_ON_TIME").any():
    print("TIMING LIMITATION: some included forecasts are scored by Arena but lack independent timezone-aware SERVER acceptance-time certification.")
if not strict_eval_df.empty:
    strict_p, strict_y = arrays_from_rows(strict_eval_df)
    print("STRICT PROVENANCE SENSITIVITY (native records only):")
    display(pd.DataFrame([score_probabilities(strict_p,strict_y)]))
else:
    print("STRICT PROVENANCE SENSITIVITY: n=0, no performance score; do not substitute legacy-only statistics.")
MODEL_METRICS=None
if eval_df.empty:
    print("NO ELIGIBLE OFFICIALLY SETTLED RECORDS YET — no real performance statistics calculated.")
else:
    p,y=arrays_from_rows(eval_df)
    MODEL_METRICS=score_probabilities(p,y)
    display(pd.DataFrame([{"metric":k,"value":v} for k,v in MODEL_METRICS.items()]))
    cm=pd.crosstab(pd.Categorical(eval_df.official_label,categories=CLASSES),
                   pd.Categorical(eval_df.predicted_label,categories=CLASSES),dropna=False)
    cm.index.name="Official outcome";cm.columns.name="Model top class"
    display(cm)
    class_stats=[]
    for k,c in enumerate(CLASSES):
        actual=(y==k); chosen=(p.argmax(axis=1)==k)
        tp=int((actual&chosen).sum())
        class_stats.append({"class":c,"support":int(actual.sum()),"precision":tp/max(int(chosen.sum()),1) if chosen.sum() else np.nan,
                            "recall":tp/max(int(actual.sum()),1) if actual.sum() else np.nan})
    display(pd.DataFrame(class_stats))
    
    # Audit table: all officially settled accepted forecasts.
    daily_audit = eval_df[
        [
            "settlement_date",
            "challenge_id",
            "predicted_label",
            "official_label",
            "arena_score",
        ]
    ].copy()

    daily_audit["correct"] = (
        daily_audit["predicted_label"]
        == daily_audit["official_label"]
    )

    print("OFFICIAL SETTLED FORECASTS — CHRONOLOGICAL AUDIT")
    display(
        daily_audit.sort_values("settlement_date")
                   .reset_index(drop=True)
    )

    print(f"INTERIM: {len(eval_df)}/{COHORT_TARGET_N} descriptively qualified accepted-and-settled; strict UTC-certified native n={len(strict_eval_df)}. No demonstrated skill claim.")
    if "accepted_by_external_evidence" in eval_df.columns and eval_df.accepted_by_external_evidence.fillna(False).any():
        native_only=eval_df.loc[~eval_df.accepted_by_external_evidence.fillna(False)].copy()
        if native_only.empty: print("Legacy sensitivity: no native settled observations yet")
        else:
            q,z=arrays_from_rows(native_only)
            print("Sensitivity excluding recovered legacy submission (NOT the primary cohort):")
            display(pd.DataFrame([score_probabilities(q,z)]))

## 08 · Prespecified / historically documented benchmarks for the same descriptive challenge subset

Use a uniform three-class distribution and clearly labeled training-only class counts. The original baseline definitions have a known 3,855/3,835 population/smoothing discrepancy; these historical comparisons are **descriptive until one primary baseline is prospectively frozen**. No evaluation-period outcomes may be used to estimate class priors. Hard always-class rules report accuracy only, not fabricated probabilities.

In [ ]:
prior=None
if not train_raw.empty:
    d=train_raw.copy()
    assert len(d)==3 and set(d.class_label)==set(CLASSES) and not d.class_label.duplicated().any(),"Training prior needs precisely 3 unique classes"
    counts=np.array([float(d.loc[d.class_label==c,"count"].iloc[0]) for c in CLASSES])
    assert np.isfinite(counts).all() and (counts>=0).all() and counts.sum()>0
    assert d.source_description.ne("").all(),"Training-only provenance required"
    prior=counts/counts.sum()
    print("Training-only priors, frozen provenance:",dict(zip(CLASSES,prior)))
else: print("TRAINING-ONLY PRIOR: unavailable; intentionally not estimated from prospective outcomes.")

comparison=[]
if not eval_df.empty:
    p,y=arrays_from_rows(eval_df)
    models={"Frozen Logistic Regression":p,"Uniform 1/3":np.full_like(p,1/3)}
    if prior is not None: models["Frozen training-only class prior"]=np.tile(prior,(len(y),1))
    
    for name, q in models.items():
        metrics = score_probabilities(q, y)

        # Uniform probabilities have no unique predicted class.
        # Do not report arbitrary argmax tie-breaking as accuracy.
        if name == "Uniform 1/3":
            metrics["accuracy"] = np.nan

        comparison.append({
            "model": name,
            **metrics,
        })

    # Perpetual one-label rules: accuracy only. Do not masquerade as calibrated probabilistic models.
    for i,c in enumerate(CLASSES):
        comparison.append({"model":f"Always {c} (hard-class, accuracy only)","n":len(y),
                           "accuracy":float(np.mean(y==i)),"avg_confidence":np.nan,"brier":np.nan,"log_loss":np.nan})
    comparison_df=pd.DataFrame(comparison)
    display(comparison_df)
    if len(y):
        print("Skill vs uniform Brier (positive = better):",round(1-MODEL_METRICS["brier"]/(2/3),4))
else:
    comparison_df=pd.DataFrame()
    print("BASELINE TABLE: pending official settled outcomes.")

## 09 · Preliminary calibration and uncertainty (descriptive subset only)

Do not infer calibration from a small number of outcomes. Wilson intervals assume independent Bernoulli observations and may be anti-conservative for a time series. The plots are diagnostic, not validation of a tradeable edge.

In [ ]:
def wilson_interval(success,total,z=1.959963984540054):
    if total==0:return (np.nan,np.nan)
    p=success/total;den=1+z*z/total
    ctr=(p+z*z/(2*total))/den
    rad=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return max(0,ctr-rad),min(1,ctr+rad)

if eval_df.empty:
    print("CALIBRATION: pending verified settled observations.")
else:
    p,y=arrays_from_rows(eval_df)
    correct=(p.argmax(axis=1)==y)
    conf=p.max(axis=1)
    lo,hi=wilson_interval(int(correct.sum()),len(correct))
    print(f"Accuracy: {correct.mean():.1%} | Wilson 95% interval: [{lo:.1%}, {hi:.1%}] | n={len(correct)}")
    print("Wilson interval assumes effectively independent Bernoulli events; time dependence may widen uncertainty.")
    bins=np.array([0,1/3,2/3,1.0000001])
    ids=np.clip(np.searchsorted(bins,conf,side="right")-1,0,2)
    cal=[]
    for i in range(3):
        m=(ids==i)
        cal.append({"confidence_bin":f"{bins[i]:.2f}–{min(1,bins[i+1]):.2f}","n":int(m.sum()),
                    "mean_confidence":float(conf[m].mean()) if m.any() else np.nan,
                    "observed_accuracy":float(correct[m].mean()) if m.any() else np.nan})
    display(pd.DataFrame(cal))
    onehot=np.eye(3)[y]
    class_cal=[]
    for j,c in enumerate(CLASSES):
        for a,b in [(0,1/3),(1/3,2/3),(2/3,1.0000001)]:
            m=(p[:,j]>=a)&(p[:,j]<b)
            class_cal.append({"class":c,"p_range":f"[{a:.2f},{min(b,1):.2f}]","n":int(m.sum()),
                              "mean_probability":float(p[m,j].mean()) if m.any() else np.nan,
                              "empirical_frequency":float(onehot[m,j].mean()) if m.any() else np.nan})
    display(pd.DataFrame(class_cal))
    fig,ax=plt.subplots(figsize=(6.4,4))
    tab=pd.DataFrame(cal).dropna(subset=["mean_confidence","observed_accuracy"])
    ax.plot([0,1],[0,1],linestyle="--",label="Perfect calibration")
    if not tab.empty: ax.scatter(tab.mean_confidence,tab.observed_accuracy,s=45,label="Observed confidence bins")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Average predicted confidence",ylabel="Fraction correct",title="Top-class calibration — preliminary")
    ax.legend();plt.show()

## 10 · External Headline Arena competitor benchmarking

**Primary peer estimand:** compare the exact same ES `challenge_id` with another agent's **valid, pre-deadline probability vector**, using the *same official outcome*. Report matched `n`, the number of our total eligible observations that have a peer, and paired changes in accuracy and unscaled Brier. An agent with fewer matched forecasts cannot be evaluated on the full cohort; no imputation of absent predictions.

**Secondary:** a public leaderboard rank/score is *context only*, because agents may cover different challenges, assets and dates. We do not treat an all-market leaderboard ranking as a head-to-head ES comparison. Peer identity and data rights must be documented.

In [ ]:
def validate_peers(raw):
    if raw.empty:return raw.copy()
    d=raw.copy()
    assert d.challenge_id.ne("").all() and d.agent_id.ne("").all(),"Missing peer IDs"
    assert not d.duplicated(["challenge_id","agent_id"]).any(),"Duplicate peer rows"
    from urllib.parse import urlparse
    assert d.apply(lambda r: urlparse(r.source_url).scheme=="https" and
                   urlparse(r.source_url).hostname in {"headlinearena.com","www.headlinearena.com"} and
                   r.challenge_id.lower() in urlparse(r.source_url).path.lower(),axis=1).all(),"Peer official HTTPS source must reference same challenge"
    assert d.counts_for_score.str.lower().eq("true").all(),"Peer must be recorded as scoring-eligible; skip unverified peers"
    assert d.submitted_at_utc.ne("").all(),"Peer timestamp required for comparability"
    for c in CLASSES: d[f"p_{c}"]=pd.to_numeric(d[f"p_{c}"],errors="raise")
    prob=d[[f"p_{c}" for c in CLASSES]].to_numpy(dtype=float)
    assert np.isfinite(prob).all() and (prob>=0).all() and (prob<=1).all() and np.allclose(prob.sum(axis=1),1,atol=1e-6),"Invalid peer probabilities"
    d["challenge_id"]=d.challenge_id.str.lower()
    return d

peer=validate_peers(peer_raw)
peer_results=[]
if eval_df.empty or peer.empty:
    print("MATCHED ARENA AGENT COMPARISON: unavailable until peer probabilities and official results are documented.")
else:
    joined=peer.merge(eval_df[["challenge_id","official_label","settlement_date",
                                  "p_bearish","p_neutral","p_bullish"]],
                      on="challenge_id",suffixes=("_peer","_ours"),validate="many_to_one")
    if "official_deadline_utc" in eval_df:
        dl=eval_df[["challenge_id","official_deadline_utc"]].drop_duplicates("challenge_id")
        joined=joined.merge(dl,on="challenge_id",how="left",validate="many_to_one")
        ok=[]
        for a,b in zip(joined.submitted_at_utc,joined.official_deadline_utc):
            ok.append(pd.notna(parse_utc(a)) and pd.notna(parse_utc(b)) and parse_utc(a)<=parse_utc(b))
        joined=joined.loc[ok].copy()
        if len(joined)==0: print("No peers with verified on-time challenge-matched submissions")
    else:
        print("WARNING: No official deadlines in outcomes CSV; peer timestamp validity unverified — comparison excluded.")
        joined=joined.iloc[0:0]
    for agent_id,d in joined.groupby("agent_id",sort=True):
        ours=d[[f"p_{c}_ours" for c in CLASSES]].to_numpy(dtype=float)
        theirs=d[[f"p_{c}_peer" for c in CLASSES]].to_numpy(dtype=float)
        y=np.array([CLASS_TO_I[t] for t in d.official_label])
        a=score_probabilities(ours,y); b=score_probabilities(theirs,y)
        peer_results.append({"agent_id":agent_id,"matched_n":len(d),"coverage_of_our_settled_pct":100*len(d)/len(eval_df),
                             "our_accuracy_matched":a["accuracy"],"peer_accuracy_matched":b["accuracy"],
                             "peer_minus_our_accuracy":b["accuracy"]-a["accuracy"],
                             "our_brier_matched":a["brier"],"peer_brier_matched":b["brier"],
                             "peer_minus_our_brier":b["brier"]-a["brier"]})
    if peer_results: display(pd.DataFrame(peer_results).sort_values(["matched_n","peer_brier_matched"],ascending=[False,True]))
    else: print("No comparable matched peers; this is NOT a competitor-performance conclusion.")

## 11 · Local Databento ↔ Arena settlement-target reconciliation

This is a **label-alignment audit**, not the official scoring source. The local ES session uses Databento's 18:00–17:00 ET minute-bar construction. Arena may sample a slightly different settlement price. We compare returns (percent units) and the implied bullish/neutral/bearish labels for **the matching challenge settlement date**, and flag mismatches or proximity to ±0.3 percentage points.

Overlapping immutable snapshots are cross-checked; conflicting historical records cause a stop. If a snapshot is unavailable, mark local comparison missing—not a zero difference. This section does not edit old labels or model features.

In [ ]:
def class_from_pct(pct):
    if pd.isna(pct): return None
    if pct>DEAD_ZONE_PCT:return "bullish"
    if pct< -DEAD_ZONE_PCT:return "bearish"
    return "neutral"

local_rows=[]
for path in sorted(SNAPSHOT_DIR.glob("es_sessions_*.csv")) if SNAPSHOT_DIR.is_dir() else []:
    try:
        d=pd.read_csv(path)
        assert {"session_date","session_return","open","close","raw_symbol","coverage_reviewed"}.issubset(d.columns)
        for row in d.itertuples(index=False):
            local_rows.append({"session_date":str(row.session_date),"open":float(row.open),"close":float(row.close),
                               "local_return_pct":float(row.session_return)*100,"raw_symbol":str(row.raw_symbol),
                               "coverage_reviewed":str(row.coverage_reviewed).lower()=="true",
                               "snapshot_file":path.name})
    except Exception as exc:
        warnings.warn(f"Unable to parse local snapshot {path.name}: {exc}")
local_df=pd.DataFrame(local_rows)
if not local_df.empty:
    for date,group in local_df.groupby("session_date"):
        columns=["open","close","local_return_pct","raw_symbol"]
        if len(group)>1:
            assert all(np.isclose(group[x].astype(float).max(),group[x].astype(float).min(),rtol=0,atol=1e-8)
                       if x!="raw_symbol" else group[x].nunique()==1 for x in columns),f"Historical snapshot drift: {date}"
    local_df=local_df.sort_values("snapshot_file").drop_duplicates("session_date",keep="first")
    local_df["local_label"]=local_df.local_return_pct.map(class_from_pct)
    print("Cross-snapshot overlap validation: PASS; unique sessions:",len(local_df))
else: print("No validated local snapshots available for settlement comparison.")

if not eval_df.empty and not local_df.empty:
    target_check=eval_df.merge(local_df,left_on="settlement_date",right_on="session_date",how="left",validate="many_to_one")
    target_check["label_agrees"]=target_check.official_label==target_check.local_label
    target_check["return_diff_pp"]=np.nan
    if "settlement_return_pct" in target_check:
        target_check["return_diff_pp"]=target_check.local_return_pct-target_check.settlement_return_pct
    target_check["distance_to_threshold_pp"]=(target_check.local_return_pct.abs()-DEAD_ZONE_PCT).abs()
    columns=["settlement_date","challenge_id","official_label","local_label","label_agrees","local_return_pct","return_diff_pp","distance_to_threshold_pp"]
    display(target_check[columns])
    print("Official/local class disagreements (only known local rows):",int((target_check.local_label.notna() & ~target_check.label_agrees).sum()))
else: print("Target reconciliation awaits both official outcomes and local snapshots.")

## 12 · Uncertainty sensitivity, only if the sample is large enough

IID bootstrap cannot establish credible time-dependent uncertainty or a market edge with a very small sample. **Suppress bootstrap intervals for fewer than 30 descriptive observations.** At n≥30, report an expressly exploratory IID diagnostic with dependence and sample-size warnings; the future 60+60 protocol should specify block-bootstrap assumptions before Phase B.

In [ ]:
def iid_bootstrap_intervals(p,y,iterations=3000,seed=BOOTSTRAP_SEED):
    rng=np.random.default_rng(seed)
    n=len(y)
    if n<30: return None
    picks=rng.integers(0,n,size=(iterations,n))
    correct=(p.argmax(axis=1)==y).astype(float)
    ones=np.eye(3)[y]
    row_brier=((p-ones)**2).sum(axis=1)
    return {
        "accuracy_2.5%":float(np.quantile(correct[picks].mean(axis=1),.025)),
        "accuracy_97.5%":float(np.quantile(correct[picks].mean(axis=1),.975)),
        "brier_2.5%":float(np.quantile(row_brier[picks].mean(axis=1),.025)),
        "brier_97.5%":float(np.quantile(row_brier[picks].mean(axis=1),.975)),
    }
if eval_df.empty or len(eval_df)<30:
    print(f"BOOTSTRAP SUPPRESSED: n={len(eval_df)} < 30; exploratory bootstrap on an incomplete early cohort is withheld.")
else:
    p,y=arrays_from_rows(eval_df)
    print("EXPLORATORY iid bootstrap (n >= 30): ignores possible time dependence; NOT a confirmatory interval:")
    display(pd.DataFrame([iid_bootstrap_intervals(p,y)]))
    if len(y)<30: print("EARLY SAMPLE — do not interpret these intervals as evidence of model skill")

## 13 · Two separate coverage gates: descriptive forecasting vs. strict provenance certification

The main sample can be descriptive even if a server provides no timezone-aware `created_at`. **Full local-chain certification cannot be claimed** without independently supported UTC timing. The analysis is final only when the original first-30 accepted opportunity cohort is fully observed and accounted for. Missing/inconsistent evidence stays in the cohort and blocks an unqualified final claim. The October 20 exception remains disclosed.

In [ ]:
accepted_n=int(ledger.loc[(ledger.settlement_date>=COHORT_START)&ledger.accepted_with_evidence].shape[0]) if not ledger.empty else 0
settled_descriptive_n=int(full.eligible_for_descriptive_metrics.sum()) if not full.empty else 0
settled_strict_native_n=int(full.eligible_for_strict_provenance_metrics.sum()) if not full.empty else 0
first_challenge_evidenced=(not ledger.empty and ORIGINAL_FIRST_CHALLENGE_ID in set(ledger.loc[ledger.accepted_with_evidence,"challenge_id"]))
model_verified=bool(LOCAL_MODEL_VERIFIED)
first30=full.loc[full.in_first_30_accepted.fillna(False)].copy() if not full.empty else pd.DataFrame()
accepted_first30=int(first30.accepted_with_evidence.sum()) if not first30.empty else 0
observed_official_settlements=int(first30.settled_confirmed.sum()) if not first30.empty else 0
native_cohort=first30.loc[~first30.accepted_by_external_evidence.fillna(False).astype(bool)].copy() if not first30.empty else pd.DataFrame()
strict_native_coverage_n=int(native_cohort.strict_native_provenance.fillna(False).sum()) if not native_cohort.empty else 0
unresolved_native=int((provenance_df.evidence_type.eq("NATIVE") & provenance_df.provenance_status.isin(["FAIL","INCOMPLETE"])).sum()) if not provenance_df.empty else 0
fail_native=int((provenance_df.evidence_type.eq("NATIVE") & provenance_df.provenance_status.eq("FAIL")).sum()) if not provenance_df.empty else 0
legacy_limit=bool(first30.accepted_by_external_evidence.fillna(False).any()) if not first30.empty else False

# Descriptive completed cohort is NOT the same as fully independently certified chain.
# The following gate requires all 30 originally accepted slots, all settled with
# adequate scoring evidence; this does not remove accepted legacy or timezone limitations.
descriptive_cohort_complete=(accepted_first30==COHORT_TARGET_N and
    observed_official_settlements==COHORT_TARGET_N and
    settled_descriptive_n==COHORT_TARGET_N and
    not forecast_errors and not receipt_errors and not legacy_errors and fail_native==0)
strict_full_chain_complete=(descriptive_cohort_complete and not legacy_limit and
    strict_native_coverage_n==COHORT_TARGET_N and unresolved_native==0)

quality_report={
 "planned_accepted_cohort":COHORT_TARGET_N,
 "native_forecasts_found":len(forecast_paths),
 "native_forecasts_parsed":len(forecasts),
 "native_forecast_errors":len(forecast_errors),
 "quarantined_server_accepted_records":len(quarantined),
 "native_receipts_found":len(receipt_paths),
 "legacy_accepted_records":len(legacy_rows),
 "original_oct20_forecast_evidenced":bool(first_challenge_evidenced),
 "legacy_original_model_bytes_not_proven":bool(legacy_limit),
 "unrecognized_receipt_files":len(receipt_errors),
 "unrecognized_legacy_files":len(legacy_errors),
 "accepted_records_seen_since_start":accepted_n,
 "accepted_slots_in_first30_seen":accepted_first30,
 "officially_settled_first30_slots":observed_official_settlements,
 "descriptive_scored_and_settled_first30":settled_descriptive_n,
 "strict_local_chain_certified_native_settled":settled_strict_native_n,
 "strict_native_certified_within_first30":strict_native_coverage_n,
 "native_audits_incomplete_or_failed":unresolved_native,
 "native_audits_failed":fail_native,
 "frozen_model_hash_locally_verified":model_verified,
 "official_settlement_rows_available_locally":len(arena),
 "peer_rows_available_locally":len(peer),
 "descriptive_30_complete":bool(descriptive_cohort_complete),
 "full_strict_chain_completed":bool(strict_full_chain_complete),
 "model_sha256_expected":FROZEN_MODEL_SHA256,
 "analysis_status":("COMPLETE_DESCRIPTIVE_WITH_DISCLOSED_PROVENANCE_LIMITATIONS" if descriptive_cohort_complete and not strict_full_chain_complete
    else "COMPLETE_STRICT_LOCAL_CHAIN" if strict_full_chain_complete
    else "PRELIMINARY_OR_INCOMPLETE"),
}
display(pd.DataFrame(quality_report.items(),columns=["check","result"]))
if not calendar_raw.empty:
    print("Recorded missed/failed/no-challenge opportunities (do not replace selectively):")
    display(calendar_raw.sort_values("settlement_date"))
if not descriptive_cohort_complete:
    display(Markdown("**PRELIMINARY DESCRIPTIVE STUDY.** Some first-30 accepted forecasts have not settled in local evidence, or original evidence remains missing/contradictory. Do not call this final predictive skill."))
else:
    print("DESCRIPTIVE FIRST-30 COHORT COMPLETE. Provenance limitations and October 20 exception remain separately disclosed.")
if not strict_full_chain_complete:
    print("STRICT PROVENANCE CERTIFICATION NOT COMPLETE: no claim of complete independently UTC-verified server timing or model execution.")

## 14 · Research summary — findings, denominators and evidence limitations

The descriptive summary names its sample and warns about unresolved server UTC time. Strict provenance is reported separately and never upgraded from a client receipt clock. Neither status establishes actual trading profitability or the model's historical data availability at first publication.

In [ ]:
if MODEL_METRICS is None:
    statement="No descriptively qualified, officially settled ES observations are available; no performance metrics computed."
else:
    statement=(f"PRELIMINARY DESCRIPTIVE M1: {MODEL_METRICS['n']}/{COHORT_TARGET_N} accepted and locally documented officially settled original forecasts; "
               f"accuracy {MODEL_METRICS['accuracy']:.1%}; average top probability {MODEL_METRICS['avg_confidence']:.1%}; "
               f"unscaled three-class Brier {MODEL_METRICS['brier']:.4f}; log loss {MODEL_METRICS['log_loss']:.4f}. "
               f"Strict UTC-certified native settled subset n={len(strict_eval_df)}. "
               "Legacy October 20 and timezone/data-first-availability limitations disclosed; no demonstrated market edge.")
print(statement)
print("Official outcomes are loaded from the local CSV only and may lag the Arena site. Do not equate Arena platform BSS with three-class Brier skill.")
print("No changes to frozen M1, its past submissions or original evidence. Continue original 30 accepted forecasts and prepare future 60+60 protocol separately.")

In [ ]:

# Read-only full-precision M1 native forecast ledger.
# Does not modify original forecasts, receipts or metrics.

from IPython.display import display
import pandas as pd

native_ledger = full.loc[
    full["evidence_type"].eq("NATIVE_FORECAST_PLUS_RECEIPT")
    & full["accepted_by_receipt"].fillna(False)
].copy()

native_ledger = native_ledger.sort_values("settlement_date")

cols = [
    "accepted_rank",
    "settlement_date",
    "challenge_id",
    "last_session",
    "p_bearish",
    "p_neutral",
    "p_bullish",
    "predicted_label",
    "official_label",
    "settled_confirmed",
    "original_model_sha_verified",
    "artifact_integrity_verified",
    "accepted_by_receipt",
    "strict_native_provenance",
    "provenance_status",
    "client_receipt_recorded_at_utc",
    "server_created_at_raw",
]

available = [c for c in cols if c in native_ledger.columns]

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.float_format", "{:.12f}".format,
):
    display(native_ledger[available])

print("Native accepted forecasts:", len(native_ledger))
print(
    "Native forecasts with official outcomes:",
    int(native_ledger["settled_confirmed"].sum())
)
print(
    "Native forecasts without official outcomes:",
    int((~native_ledger["settled_confirmed"]).sum())
)
print(
    "Strictly certified native forecasts:",
    int(native_ledger["strict_native_provenance"].sum())
)
